# Are You Actually Replaceable? What the Data Says About AI and Your Career
### Design
**Group P33** — Mika Ohayon, Harini Champooranan, Akshaya Somasundaram

We will be designing six interactive visuals using the Altair visualization tool. These charts will incorporate dynamic features like hover tools, sliders, click, zoom, panning, linked interactions and brushing to allow for deep exploration of the underlying data patterns. Each visual will be carefully styled with clear annotations and legends to guide the audience through key insights. We will train and test up to three machine learning models to evaluate our primary hypothesis and measure predictive accuracy.

## Project Goal

This project tests whether occupations already flagged as highly “exposed” to AI  using the AIOE (AI Occupational Exposure) score from Felten, Raj, and Seamans  actually show the wage and employment shifts that AI-displacement narratives predict. We will link each occupation’s AIOE score to its BLS OEWS employment/wage figures by SOC code, then run a linear regression (with a skill/education interaction term) to test whether AIOE predicts real outcomes, and whether that relationship differs for high- versus low-skill occupations. The interaction term matters because the AI-displacement narrative is not just “high exposure means job loss”,  it is often specifically that high-skill, high-exposure roles behave differently from low-skill, high-exposure ones, so a model without that interaction would blur two potentially opposite effects into one misleading average. With the data cleaned and merged, this phase is about designing the argument. We will build six interactive visuals with hover, linked brushing, zoom, and sliders, each paired with annotations and a legend, and each carrying one central point of the story. We will test our claims with up to three machine learning models, so that the evidence is both seen and statistically supported.

In [1]:
# import libraries

import numpy as np
import pandas as pd
import altair as alt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import permutation_importance

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [2]:
# suppress warnings 
import warnings
warnings.filterwarnings('ignore')

In [3]:
# read data files and create a data frames
df1 = pd.read_csv ('./dataset1_new.csv')
df2 = pd.read_csv ('./dataset2_new.csv')

In [4]:
df1.shape

(1401, 28)

In [5]:
df1.head(5)

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,24.51,38.71,61.81,31200.0,37590.0,50980.0,80520.0,128560.0,False,False
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,60.83,84.75,123.71,60130.0,82970.0,126520.0,176280.0,257310.0,False,False
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,52.17,82.85,134.48,50480.0,73260.0,108510.0,172330.0,279710.0,False,False
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,102.88,171.25,244.10,75700.0,129540.0,213990.0,356200.0,507730.0,False,False
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,102.88,171.25,244.10,75700.0,129540.0,213990.0,356200.0,507730.0,False,False


In [6]:
df1.tail(5)

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
1396,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7081,Refuse and Recyclable Material Collectors,...,23.89,29.82,36.49,33590.0,39080.0,49690.0,62020.0,75900.0,False,False
1397,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7120,"Tank Car, Truck, and Ship Loaders",...,28.30,33.69,39.98,39460.0,46300.0,58870.0,70060.0,83160.0,False,False
1398,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7121,"Tank Car, Truck, and Ship Loaders",...,28.30,33.69,39.98,39460.0,46300.0,58870.0,70060.0,83160.0,False,False
1399,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7190,Miscellaneous Material Moving Workers,...,20.10,24.14,30.20,34330.0,36600.0,41800.0,50220.0,62820.0,False,False
1400,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7199,"Material Moving Workers, All Other",...,20.10,24.14,30.20,34330.0,36600.0,41800.0,50220.0,62820.0,False,False


In [7]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 1401 entries, 0 to 1400
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   AREA         1401 non-null   int64  
 1   AREA_TITLE   1401 non-null   str    
 2   AREA_TYPE    1401 non-null   int64  
 3   PRIM_STATE   1401 non-null   str    
 4   NAICS        1401 non-null   int64  
 5   NAICS_TITLE  1401 non-null   str    
 6   I_GROUP      1401 non-null   str    
 7   OWN_CODE     1401 non-null   int64  
 8   OCC_CODE     1401 non-null   str    
 9   OCC_TITLE    1401 non-null   str    
 10  O_GROUP      1401 non-null   str    
 11  TOT_EMP      1401 non-null   int64  
 12  EMP_PRSE     1401 non-null   float64
 13  H_MEAN       1320 non-null   float64
 14  A_MEAN       1394 non-null   float64
 15  MEAN_PRSE    1401 non-null   float64
 16  H_PCT10      1320 non-null   float64
 17  H_PCT25      1320 non-null   float64
 18  H_MEDIAN     1320 non-null   float64
 19  H_PCT75      1320

In [8]:
df1.describe()

,AREA,AREA_TYPE,NAICS,OWN_CODE,TOT_EMP,EMP_PRSE,H_MEAN,A_MEAN,MEAN_PRSE,H_PCT10,H_PCT25,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90
count,1401.0,1401.0,1401.0,1401.0,1.401000e+03,1401.000000,1320.000000,1394.000000,1401.00000,1320.000000,1320.000000,1320.000000,1320.000000,1320.000000,1394.000000,1394.000000,1394.000000,1394.000000,1394.000000
mean,99.0,1.0,0.0,1235.0,5.549249e+05,2.954318,37.152273,78378.271162,1.10828,21.441348,26.704242,34.562106,44.270917,55.944705,44912.517934,56067.611191,72615.272597,93363.802009,118870.516499
std,0.0,0.0,0.0,0.0,4.342791e+06,3.569802,22.596543,46831.664875,1.47156,8.658766,13.620381,22.599702,29.644689,38.472242,17773.882516,28006.862647,46327.286155,61417.913805,81738.114316
min,99.0,1.0,0.0,1235.0,2.200000e+02,0.000000,15.370000,31980.000000,0.00000,8.700000,12.410000,14.850000,17.090000,18.390000,18100.000000,25810.000000,30890.000000,35550.000000,38240.000000
25%,99.0,1.0,0.0,1235.0,2.043000e+04,0.900000,23.657500,49937.500000,0.40000,16.165000,18.350000,22.300000,27.715000,32.920000,33877.500000,38370.000000,46800.000000,58252.500000,69965.000000
50%,99.0,1.0,0.0,1235.0,6.999000e+04,1.800000,30.590000,65450.000000,0.60000,18.510000,22.535000,28.335000,36.525000,45.430000,39000.000000,47650.000000,60160.000000,77500.000000,97575.000000
75%,99.0,1.0,0.0,1235.0,2.833800e+05,3.600000,42.937500,91290.000000,1.20000,23.750000,30.110000,38.597500,50.920000,65.700000,49852.500000,63400.000000,81520.000000,106920.000000,139845.000000
max,99.0,1.0,0.0,1235.0,1.554957e+08,35.000000,241.370000,502050.000000,15.90000,103.560000,161.720000,268.760000,293.300000,349.350000,215410.000000,336380.000000,559030.000000,610060.000000,742310.000000


In [9]:
df2.shape

(774, 3)

In [10]:
df2.head(5)

,SOC Code,Occupation Title,AIOE
0,11-1011,Chief Executives,1.334246
1,11-1021,General and Operations Managers,0.574877
2,11-2011,Advertising and Promotions Managers,1.294387
3,11-2021,Marketing Managers,1.315032
4,11-2022,Sales Managers,1.266280


In [11]:
df2.tail(5)

,SOC Code,Occupation Title,AIOE
769,53-7072,"Pump Operators, Except Wellhead Pumpers",-0.842293
770,53-7073,Wellhead Pumpers,-1.091197
771,53-7081,Refuse and Recyclable Material Collectors,-1.283373
772,53-7111,Shuttle Car Operators,-1.351998
773,53-7121,"Tank Car, Truck, and Ship Loaders",-1.387760


In [12]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 774 entries, 0 to 773
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   SOC Code          774 non-null    str    
 1   Occupation Title  774 non-null    str    
 2   AIOE              774 non-null    float64
dtypes: float64(1), str(2)
memory usage: 18.3 KB


In [13]:
df2.describe()

,AIOE
count,7.740000e+02
mean,-6.847545e-09
std,1.000000e+00
min,-2.669758e+00
25%,-8.645451e-01
50%,-5.119285e-02
75%,1.014434e+00
max,1.527667e+00


In [14]:
MAJOR_NAMES = {
    '11': 'Management', '13': 'Business & Financial', '15': 'Computer & Mathematical',
    '17': 'Architecture & Engineering', '19': 'Life, Physical & Social Science',
    '21': 'Community & Social Service', '23': 'Legal', '25': 'Education & Library',
    '27': 'Arts, Design & Media', '29': 'Healthcare Practitioners',
    '31': 'Healthcare Support', '33': 'Protective Service', '35': 'Food Prep & Serving',
    '37': 'Building & Grounds Cleaning', '39': 'Personal Care & Service', '41': 'Sales',
    '43': 'Office & Admin Support', '45': 'Farming, Fishing & Forestry',
    '47': 'Construction & Extraction', '49': 'Installation & Repair',
    '51': 'Production', '53': 'Transportation & Moving', '55': 'Military',
}

oews = df1[df1['O_GROUP'].str.lower() == 'detailed'].copy()

check = oews.merge(df2, left_on='OCC_CODE', right_on='SOC Code', how='outer', indicator=True)
print(check['_merge'].value_counts())

df = check[check['_merge'] == 'both'].copy()
df = df.dropna(subset=['A_MEDIAN', 'AIOE', 'TOT_EMP'])
df = df[df['TOT_EMP'] > 0]

HI, LO = 'Higher-skill (SOC 11–29)', 'Lower-skill (SOC 31–53)'
df['soc_major'] = df['OCC_CODE'].str[:2]
df['major_group'] = df['soc_major'].map(MAJOR_NAMES).fillna(df['soc_major'])
df['high_skill'] = (df['soc_major'].astype(int) <= 29).astype(int)
df['skill_tier'] = np.where(df['high_skill'] == 1, HI, LO)
df['AIOE_x_skill'] = df['AIOE'] * df['high_skill']          
df['log_wage'] = np.log(df['A_MEDIAN'])                     
df['log_emp'] = np.log10(df['TOT_EMP'])

BAND_LABELS = ['Bottom 20%', '20–40%', '40–60%', '60–80%', 'Top 20%']
BAND_ORDER = BAND_LABELS[::-1]
df['wage_band'] = pd.qcut(df['A_MEDIAN'].rank(method='first'), 5, labels=BAND_LABELS).astype(str)
df['aioe_bin_start'] = np.floor(df['AIOE'] / 0.5) * 0.5
df['aioe_bin'] = df['aioe_bin_start'].map(lambda s: f'{s:+.1f} to {s + 0.5:+.1f}')
BIN_ORDER = [f'{s:+.1f} to {s + 0.5:+.1f}' for s in sorted(df['aioe_bin_start'].unique())]

df = df.rename(columns={'OCC_TITLE': 'occupation'})
df = df[['OCC_CODE', 'occupation', 'soc_major', 'major_group', 'skill_tier', 'high_skill',
         'AIOE', 'AIOE_x_skill', 'TOT_EMP', 'log_emp', 'A_MEDIAN', 'log_wage',
         'A_PCT10', 'A_PCT25', 'A_PCT75', 'A_PCT90',
         'wage_band', 'aioe_bin_start', 'aioe_bin']].reset_index(drop=True)
print(df.shape)

_merge
both          673
left_only     157
right_only    101
Name: count, dtype: int64
(670, 19)


In [15]:
tier_scale = alt.Scale(domain=[HI, LO], range=["#0d3eb9", "#d9116bff"])   # blue = higher, orange = lower
GREY = '#d4d4d4'

def tier_color(legend=True):
    return alt.Color('skill_tier:N', scale=tier_scale,
                     legend=alt.Legend(title='Skill tier') if legend else None)

occ_tip = [
    alt.Tooltip('occupation:N', title='Occupation'),
    alt.Tooltip('major_group:N', title='Group'),
    alt.Tooltip('AIOE:Q', title='AIOE', format='.2f'),
    alt.Tooltip('A_MEDIAN:Q', title='Median wage', format='$,.0f'),
    alt.Tooltip('TOT_EMP:Q', title='Workers', format=',.0f'),
]

### Interactive Visual 1 : 

In [16]:
group_stats = (df.groupby(['major_group', 'skill_tier'], as_index=False)
                 .agg(mean_aioe=('AIOE', 'mean'), workers=('TOT_EMP', 'sum'), n_occ=('AIOE', 'size')))

group_click = alt.selection_point(fields=['major_group'])

bars = (alt.Chart(group_stats).mark_bar().encode(
    y=alt.Y('major_group:N', sort='-x', title=None, axis=alt.Axis(labelLimit=220)),
    x=alt.X('mean_aioe:Q', title='Average AIOE exposure (standardized)'),
    color=alt.condition(group_click, tier_color(True), alt.value(GREY)),
    tooltip=[alt.Tooltip('major_group:N', title='Occupation group'),
             alt.Tooltip('mean_aioe:Q', title='Mean AIOE', format='.2f'),
             alt.Tooltip('n_occ:Q', title='Occupations'),
             alt.Tooltip('workers:Q', title='Workers', format=',.0f')],
).add_params(group_click).properties(width=360, height=440,
                                     title='Click a group'))

zero = pd.DataFrame({'x': [0], 'label': ['Average occupation (AIOE = 0)']})
zero_rule = alt.Chart(zero).mark_rule(color='black', strokeDash=[4, 3]).encode(x='x:Q')
zero_text = (alt.Chart(zero).mark_text(align='left', baseline='top', dx=4, dy=2, fontSize=11)
             .encode(x='x:Q', y=alt.value(0), text='label:N'))

hist = (alt.Chart(df).mark_bar().encode(
    x=alt.X('AIOE:Q', bin=alt.Bin(maxbins=25), title='AIOE of occupations in the selected group'),
    y=alt.Y('count():Q', title='Number of occupations'),
    color=tier_color(False),
    tooltip=[alt.Tooltip('count():Q', title='Occupations')],
).transform_filter(group_click).properties(width=320, height=440,
                                           title='How exposure is spread inside the selection'))

visual1 = alt.layer(bars, zero_rule, zero_text) | hist
visual1

alt.HConcatChart(...)

### Interactive Visual 2 :

In [17]:
zoom_pan = alt.selection_interval(bind='scales', encodings=['x', 'y'])      
legend_sel = alt.selection_point(fields=['skill_tier'], bind='legend')      
size_var = alt.param(value=60, bind=alt.binding_range(min=10, max=250, step=10, name='Point size: '))

points = (alt.Chart(df).mark_circle(size=size_var).encode(
    x=alt.X('AIOE:Q', title='AI Occupational Exposure (AIOE, standardized)'),
    y=alt.Y('A_MEDIAN:Q', scale=alt.Scale(type='log'), title='Median annual wage (USD, log scale)'),
    color=tier_color(True),
    opacity=alt.condition(legend_sel, alt.value(0.75), alt.value(0.08)),
    tooltip=occ_tip,
).add_params(zoom_pan, legend_sel, size_var))

fit_rows, label_rows = [], []
for tier, g in df.groupby('skill_tier'):
    slope, intercept = np.polyfit(g['AIOE'], g['log_wage'], 1)
    for x in (g['AIOE'].min(), g['AIOE'].max()):
        fit_rows.append({'skill_tier': tier, 'AIOE': x, 'A_MEDIAN': np.exp(intercept + slope * x)})
    x_end = g['AIOE'].max()
    label_rows.append({'skill_tier': tier, 'AIOE': x_end, 'A_MEDIAN': np.exp(intercept + slope * x_end),
                       'label': f'{(np.exp(slope) - 1) * 100:+.0f}% wage per +1 SD AIOE'})
fit_df, label_df = pd.DataFrame(fit_rows), pd.DataFrame(label_rows)

y_log = alt.Y('A_MEDIAN:Q', scale=alt.Scale(type='log'))
fit_lines = (alt.Chart(fit_df).mark_line(size=3, clip=True)
             .encode(x='AIOE:Q', y=y_log, color=tier_color(False)))
fit_labels = (alt.Chart(label_df).mark_text(align='right', dy=-12, fontWeight='bold', clip=True)
              .encode(x='AIOE:Q', y=y_log, text='label:N', color=tier_color(False)))


def callout(data, prefix, align, dx):
    return (alt.Chart(data).transform_calculate(label=f"'{prefix}' + datum.occupation")
            .mark_text(align=align, dx=dx, dy=-8, fontSize=11, fontStyle='italic', clip=True)
            .encode(x='AIOE:Q', y=y_log, text='label:N'))


most = callout(df.loc[[df['AIOE'].idxmax()]], 'Most exposed: ', 'right', -6)
least = callout(df.loc[[df['AIOE'].idxmin()]], 'Least exposed: ', 'left', 6)

visual2 = alt.layer(points, fit_lines, fit_labels, most, least).properties(
    width=720, height=460,
    title=alt.TitleParams('Does exposure pay the same in both skill tiers?',
                          subtitle=['Scroll to zoom · drag to pan · click the legend to isolate a tier · '
                                    'slider changes point size · lines are fitted on log wage']))
visual2

alt.LayerChart(...)

### Interactive Visual 3 :

In [18]:
brush = alt.selection_interval()

sc_wage = (alt.Chart(df).mark_circle(size=55).encode(
    x=alt.X('AIOE:Q', title='AIOE'),
    y=alt.Y('A_MEDIAN:Q', scale=alt.Scale(type='log'), title='Median annual wage (USD, log)'),
    color=alt.condition(brush, tier_color(True), alt.value(GREY)),
    tooltip=occ_tip,
).add_params(brush).properties(width=340, height=330, title='Drag a box here (AIOE vs wage)'))

sc_emp = (alt.Chart(df).mark_circle(size=55).encode(
    x=alt.X('AIOE:Q', title='AIOE'),
    y=alt.Y('TOT_EMP:Q', scale=alt.Scale(type='log'), title='Workers (log)'),
    color=alt.condition(brush, tier_color(False), alt.value(GREY)),
    tooltip=occ_tip,
).properties(width=340, height=330, title='The same occupations (AIOE vs employment)'))

emp_bar = (alt.Chart(df).mark_bar().encode(
    y=alt.Y('skill_tier:N', sort=[HI, LO], title=None, axis=alt.Axis(labelLimit=220)),
    x=alt.X('sum(TOT_EMP):Q', title='Workers inside the brushed region'),
    color=tier_color(False),
).transform_filter(brush))
emp_text = (emp_bar.mark_text(align='left', dx=4, fontWeight='bold')
            .encode(text=alt.Text('sum(TOT_EMP):Q', format=',.0f'), color=alt.value('black')))
emp_chart = (emp_bar + emp_text).properties(width=680, height=110,
                                            title='How many workers are in the brushed region?')

visual3 = ((sc_wage | sc_emp) & emp_chart).properties(
    title=alt.TitleParams('Who sits in the high-exposure, high-wage corner?',
                          subtitle=['Brush the upper-right of 3a: matching points light up in 3b '
                                    'and 3c totals the workers. Grey = outside the brush.']))
visual3

alt.VConcatChart(...)

### Interactive Visual 4 :

In [19]:
n_param = alt.param(value=8, bind=alt.binding_range(min=3, max=15, step=1, name='Show top / bottom N: '))
occ_click = alt.selection_point(fields=['occupation'], empty=False)


def ranked(order, title):
    return (alt.Chart(df)
            .transform_window(rank='rank()', sort=[alt.SortField('AIOE', order=order)])
            .transform_filter(alt.datum.rank <= n_param)
            .mark_bar().encode(
                x=alt.X('AIOE:Q', title='AIOE'),
                y=alt.Y('occupation:N', sort='-x' if order == 'descending' else 'x',
                        title=None, axis=alt.Axis(labelLimit=260)),
                color=tier_color(order == 'descending'),
                stroke=alt.condition(occ_click, alt.value('black'), alt.value('transparent')),
                strokeWidth=alt.value(2.5),
                tooltip=occ_tip)
            .add_params(occ_click)
            .properties(width=300, height=alt.Step(22), title=title))


most_exposed = ranked('descending', 'Most exposed occupations')
least_exposed = ranked('ascending', 'Least exposed occupations')

pct_order = ['A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90']
detail_base = (alt.Chart(df).transform_filter(occ_click)
               .transform_fold(pct_order, as_=['percentile', 'wage']))
x_pct = alt.X('percentile:N', sort=pct_order, title=None,
              axis=alt.Axis(labelAngle=0,
                            labelExpr="{'A_PCT10':'10th','A_PCT25':'25th','A_MEDIAN':'Median',"
                                      "'A_PCT75':'75th','A_PCT90':'90th'}[datum.label]"))
detail_line = detail_base.mark_line(point=True, color='#444').encode(
    x=x_pct, y=alt.Y('wage:Q', title='Annual wage (USD)'),
    tooltip=[alt.Tooltip('occupation:N', title='Occupation'),
             alt.Tooltip('percentile:N', title='Percentile'),
             alt.Tooltip('wage:Q', title='Wage', format='$,.0f')])
detail_label = (detail_base.transform_filter(alt.datum.percentile == 'A_MEDIAN')
                .mark_text(dy=-16, fontWeight='bold', fontSize=12)
                .encode(x=x_pct, y='wage:Q', text='occupation:N'))
detail = (detail_line + detail_label).properties(
    width=640, height=220, title='Click any bar above to see that job’s wage range')

visual4 = alt.vconcat(alt.hconcat(most_exposed, least_exposed), detail).add_params(n_param)
visual4

alt.VConcatChart(...)

### Interactive Visual 5 :

In [20]:
cut_param = alt.param(value=0.5, bind=alt.binding_range(
    min=-1.5, max=1.5, step=0.1, name='"High exposure" = AIOE above: '))
exp_scale = alt.Scale(domain=['Lower exposure', 'High exposure'], range=["#2b26ba", "#07986d"])
base5 = alt.Chart(df).transform_calculate(
    exposure=alt.expr.if_(alt.datum.AIOE > cut_param, 'High exposure', 'Lower exposure'))


def tier_exposure(y_field, title, fmt, legend):
    bars_ = base5.mark_bar().encode(
        x=alt.X('skill_tier:N', sort=[HI, LO], title=None, axis=alt.Axis(labelLimit=200, labelAngle=0)),
        xOffset=alt.XOffset('exposure:N', sort=['Lower exposure', 'High exposure']),
        y=alt.Y(y_field, title=title),
        color=alt.Color('exposure:N', scale=exp_scale,
                        legend=alt.Legend(title='Exposure group') if legend else None),
        tooltip=[alt.Tooltip('skill_tier:N', title='Skill tier'),
                 alt.Tooltip('exposure:N', title='Exposure'),
                 alt.Tooltip('count():Q', title='Occupations'),
                 alt.Tooltip(y_field, title=title, format=fmt)])
    labels_ = (bars_.mark_text(dy=-6, fontSize=11)
               .encode(text=alt.Text(y_field, format=fmt), color=alt.value('black')))
    return bars_ + labels_


wage_panel = tier_exposure('median(A_MEDIAN):Q', 'Median occupation wage (USD)', '$,.0f', True).properties(
    width=300, height=320, title='Typical wage')
emp_panel = tier_exposure('sum(TOT_EMP):Q', 'Total workers', ',.0f', False).properties(
    width=300, height=320, title='Workers in each group')

visual5 = alt.hconcat(wage_panel, emp_panel).add_params(cut_param).properties(
    title=alt.TitleParams('Is the story robust to how “high exposure” is defined?',
                          subtitle=['Drag the slider to move the exposure cutoff; bars and labels update live.']))
visual5

alt.HConcatChart(...)

### Interactive Visual 6 :

In [21]:
cell_click = alt.selection_point(fields=['aioe_bin', 'wage_band'], on='click')

heat = (alt.Chart(df).mark_rect(stroke='white', strokeWidth=1).encode(
    x=alt.X('aioe_bin:O', sort=BIN_ORDER, title='AIOE exposure band (low → high)',
            axis=alt.Axis(labelAngle=-40)),
    y=alt.Y('wage_band:O', sort=BAND_ORDER, title='Median-wage band (quintiles of occupations)'),
    color=alt.Color('sum(TOT_EMP):Q', scale=alt.Scale(scheme='blues', type='sqrt'),
                    title='Workers', legend=alt.Legend(format='~s')),
    opacity=alt.condition(cell_click, alt.value(1), alt.value(0.4)),
    tooltip=[alt.Tooltip('aioe_bin:O', title='AIOE band'),
             alt.Tooltip('wage_band:O', title='Wage band'),
             alt.Tooltip('sum(TOT_EMP):Q', title='Workers', format=',.0f'),
             alt.Tooltip('count():Q', title='Occupations')],
).add_params(cell_click).properties(
    width=380, height=300,
    title=alt.TitleParams('Click a cell',
                          subtitle=['Top-right = high pay, high exposure · bottom-left = low pay, low exposure'])))

top_occ = (alt.Chart(df).transform_filter(cell_click)
           .transform_window(rank='rank()', sort=[alt.SortField('TOT_EMP', order='descending')])
           .transform_filter(alt.datum.rank <= 8)
           .mark_bar().encode(
               x=alt.X('TOT_EMP:Q', title='Workers'),
               y=alt.Y('occupation:N', sort='-x', title=None, axis=alt.Axis(labelLimit=280)),
               color=tier_color(True),
               tooltip=occ_tip)
           .properties(width=320, height=300,
                       title='Largest occupations in the clicked cell (all cells until you click)'))

visual6 = heat | top_occ
visual6


alt.HConcatChart(...)

## Machine Learning Models - Visual 7

In [22]:
FEATURES = ['AIOE', 'high_skill', 'AIOE_x_skill', 'log_emp']
FEATURE_LABELS = {'AIOE': 'AIOE', 'high_skill': 'Higher-skill (0/1)',
                  'AIOE_x_skill': 'AIOE × higher-skill', 'log_emp': 'log10 employment'}

X, y = df[FEATURES], df['log_wage']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

models = {
    'OLS (with interaction)': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                   max_depth=3, random_state=42),
}

metric_rows, imp_rows, pred_frames = [], [], []
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    p_tr, p_te = mdl.predict(X_train), mdl.predict(X_test)
    for split, yt, yp in (('Train', y_train, p_tr), ('Test', y_test, p_te)):
        metric_rows.append({'model': name, 'split': split, 'r2': r2_score(yt, yp),
                            'rmse': float(np.sqrt(mean_squared_error(yt, yp)))})
    pi = permutation_importance(mdl, X_test, y_test, n_repeats=20, random_state=42, scoring='r2')
    for f, m in zip(FEATURES, pi.importances_mean):
        imp_rows.append({'model': name, 'feature': FEATURE_LABELS[f], 'importance': m})
    frame = df.loc[X_test.index, ['occupation', 'skill_tier', 'AIOE']].copy()
    frame['model'] = name
    frame['actual'] = np.exp(y_test.values)
    frame['predicted'] = np.exp(p_te)
    pred_frames.append(frame)

metrics_df, imp_df, pred_df = pd.DataFrame(metric_rows), pd.DataFrame(imp_rows), pd.concat(pred_frames)
print(metrics_df.round(3))

ols = models['OLS (with interaction)']
coef = dict(zip(FEATURES, ols.coef_))
coef_df = pd.DataFrame({'feature': [FEATURE_LABELS[f] for f in FEATURES], 'coef': ols.coef_})
pct_lo = (np.exp(coef['AIOE']) - 1) * 100
pct_hi = (np.exp(coef['AIOE'] + coef['AIOE_x_skill']) - 1) * 100

model_names = list(models)
model_select = alt.selection_point(
    fields=['model'], bind=alt.binding_select(options=model_names, name='Model: '),
    value=[{'model': model_names[0]}])

lo_, hi_ = pred_df[['actual', 'predicted']].min().min() * 0.9, pred_df[['actual', 'predicted']].max().max() * 1.1
log_scale = alt.Scale(type='log', domain=[lo_, hi_])

pts = (alt.Chart(pred_df).transform_filter(model_select).mark_circle(size=60, opacity=0.7).encode(
    x=alt.X('actual:Q', scale=log_scale, title='Actual median wage (USD, log)'),
    y=alt.Y('predicted:Q', scale=log_scale, title='Predicted median wage (USD, log)'),
    color=tier_color(True),
    tooltip=[alt.Tooltip('occupation:N', title='Occupation'),
             alt.Tooltip('AIOE:Q', format='.2f'),
             alt.Tooltip('actual:Q', title='Actual', format='$,.0f'),
             alt.Tooltip('predicted:Q', title='Predicted', format='$,.0f')],
).add_params(model_select))
diag_df = pd.DataFrame({'x': [lo_, hi_], 'y': [lo_, hi_]})
diag = alt.Chart(diag_df).mark_line(strokeDash=[5, 4], color='black').encode(
    x=alt.X('x:Q', scale=log_scale), y=alt.Y('y:Q', scale=log_scale))
diag_lbl = (alt.Chart(pd.DataFrame({'x': [hi_ * 0.8], 'y': [hi_ * 0.8], 'label': ['Perfect prediction (y = x)']}))
            .mark_text(align='right', dy=-10, fontSize=11)
            .encode(x=alt.X('x:Q', scale=log_scale), y=alt.Y('y:Q', scale=log_scale), text='label:N'))
scatter7 = alt.layer(pts, diag, diag_lbl).properties(width=360, height=320,
                                                     title='Held-out occupations: actual vs predicted')

importance = (alt.Chart(imp_df).transform_filter(model_select).mark_bar(color="#0576cc").encode(
    x=alt.X('importance:Q', title='Permutation importance (drop in test R²)'),
    y=alt.Y('feature:N', sort='-x', title=None, axis=alt.Axis(labelLimit=200)),
    tooltip=[alt.Tooltip('feature:N', title='Feature'),
             alt.Tooltip('importance:Q', title='Importance', format='.3f')],
).properties(width=330, height=320, title='Which inputs matter for the selected model?'))

r2_bars = (alt.Chart(metrics_df).mark_bar().encode(
    x=alt.X('model:N', title=None, axis=alt.Axis(labelAngle=-15, labelLimit=150)),
    xOffset='split:N',
    y=alt.Y('r2:Q', title='R² (on log wage)'),
    color=alt.Color('split:N', scale=alt.Scale(domain=['Train', 'Test'], range=["#baff9a", "#088c21"]),
                    title='Data split'),
    opacity=alt.condition(model_select, alt.value(1), alt.value(0.35)),
    tooltip=[alt.Tooltip('model:N', title='Model'), alt.Tooltip('split:N', title='Split'),
             alt.Tooltip('r2:Q', title='R²', format='.3f'), alt.Tooltip('rmse:Q', title='RMSE (log)', format='.3f')]))
r2_text = (r2_bars.mark_text(dy=-6, fontSize=10)
           .encode(text=alt.Text('r2:Q', format='.2f'), color=alt.value('black'), opacity=alt.value(1)))
r2_chart = (r2_bars + r2_text).properties(width=360, height=260,
                                          title='Accuracy: train vs test (gap = overfitting)')

coef_bars = (alt.Chart(coef_df).mark_bar().encode(
    y=alt.Y('feature:N', sort=None, title=None, axis=alt.Axis(labelLimit=200)),
    x=alt.X('coef:Q', title='OLS coefficient (log-wage points)'),
    color=alt.condition(alt.datum.coef > 0, alt.value("#ffe643"), alt.value("#ff4b37")),
    tooltip=[alt.Tooltip('feature:N', title='Feature'), alt.Tooltip('coef:Q', title='Coefficient', format='.3f')]))
coef_zero = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='black').encode(x='x:Q')
coef_chart = (coef_bars + coef_zero).properties(
    width=330, height=260,
    title=alt.TitleParams('OLS with interaction term',
                          subtitle=[f'+1 SD AIOE → {pct_lo:+.1f}% wage (lower-skill), {pct_hi:+.1f}% (higher-skill)']))

row1 = alt.hconcat(scatter7, importance).resolve_scale(color='independent')
row2 = alt.hconcat(r2_chart, coef_chart).resolve_scale(color='independent')
visual7 = alt.vconcat(row1, row2).resolve_scale(color='independent').properties(
    title=alt.TitleParams('Does AIOE actually predict wages out of sample?',
                          subtitle=['Use the dropdown to switch models; 7a–7b update, 7c highlights the chosen model.']))
visual7

                    model  split     r2   rmse
0  OLS (with interaction)  Train  0.402  0.311
1  OLS (with interaction)   Test  0.485  0.302
2           Random Forest  Train  0.718  0.214
3           Random Forest   Test  0.460  0.309
4       Gradient Boosting  Train  0.768  0.194
5       Gradient Boosting   Test  0.443  0.314


alt.VConcatChart(...)